# llama-index-meetstream walkthrough

Every feature of the package, one step at a time, against your own MeetStream account.

**Nothing secret goes in this notebook.** Keys and meeting links are read from environment
variables. Before committing the notebook, clear its outputs:
`jupyter nbconvert --clear-output --inplace notebooks/walkthrough.ipynb`. Outputs can contain
bot ids, transcripts and participants' names.

Set these before starting Jupyter:

| Variable | Needed for |
|---|---|
| `MEETSTREAM_API_KEY` | everything |
| `MEETING_URL` | dispatching a new bot (a meeting you can admit it to) |
| `EXISTING_BOT_ID` | instead of `MEETING_URL`: a bot whose meeting has already ended |
| `OPENAI_API_KEY` | the agent and RAG cells (skipped without it) |

## 1. Install

In [ ]:
import sys
!{sys.executable} -m pip install -q -e "..[examples]"

## 2. Configuration

In [ ]:
import os

if not os.environ.get("MEETSTREAM_API_KEY"):
    raise RuntimeError("Set MEETSTREAM_API_KEY in your environment before starting Jupyter.")

MEETING_URL = os.environ.get("MEETING_URL", "")          # e.g. https://meet.google.com/abc-defg-hij
EXISTING_BOT_ID = os.environ.get("EXISTING_BOT_ID", "")
if not (MEETING_URL or EXISTING_BOT_ID):
    raise RuntimeError("Set MEETING_URL (to dispatch a bot) or EXISTING_BOT_ID (to reuse one).")

FRESH_DISPATCH = bool(MEETING_URL)
RUN_LLM_CELLS = bool(os.environ.get("OPENAI_API_KEY"))
print("Dispatching a new bot" if FRESH_DISPATCH else "Reusing an existing bot", "| LLM cells:", RUN_LLM_CELLS)

## 3. The client

`MeetStreamClient` calls MeetStream's REST API with your key. Bots are transcribed by
MeetStream's own engine unless you choose another provider, for example
`MeetStreamClient(transcription_provider="deepgram", transcription_options={"model": "nova-3"})`.

In [ ]:
from llama_index_meetstream import TRANSCRIPTION_PROVIDERS, MeetStreamAPIError, MeetStreamClient, MeetStreamReader
from llama_index_meetstream.tools import get_meetstream_tools

client = MeetStreamClient()
print("Transcription provider:", client.transcription_provider, "| known providers:", TRANSCRIPTION_PROVIDERS)

## 4. Dispatch a bot, or reuse one

In [ ]:
if FRESH_DISPATCH:
    dispatched = client.dispatch_bot(MEETING_URL, bot_name="Walkthrough Bot")
    bot_id = dispatched.meeting_id
    print(dispatched)
else:
    bot_id = EXISTING_BOT_ID
    print("Using bot", bot_id)

## 5. Meeting status

Re-run until the status shows the bot is in the meeting (admit it from the meeting first).

In [ ]:
print(client.get_meeting(bot_id))

## 6. In-meeting actions

Only while the bot is in the meeting. Skipped when reusing a finished bot.

In [ ]:
if FRESH_DISPATCH:
    print(client.send_chat_message(bot_id, "Hello from the walkthrough notebook."))
    print(client.send_image(bot_id, "https://upload.wikimedia.org/wikipedia/commons/4/47/PNG_transparency_demonstration_1.png", 10))

Talk for a minute so there's something to transcribe, then make the bot leave:

In [ ]:
if FRESH_DISPATCH:
    print(client.leave_meeting(bot_id))

## 7. Wait for the transcript

`transcript_not_ready` means MeetStream is still processing. Keep polling.

In [ ]:
import time


def wait_for_transcript(meeting_id, attempts=20, delay_seconds=15):
    for attempt in range(attempts):
        try:
            return client.get_transcript(meeting_id)
        except MeetStreamAPIError as exc:
            if exc.code != "transcript_not_ready":
                raise
            print(f"Not ready ({attempt + 1}/{attempts}); waiting {delay_seconds}s")
            time.sleep(delay_seconds)
    raise TimeoutError("Transcript still processing; run this cell again later.")


transcript = wait_for_transcript(bot_id)
print(len(transcript.segments), "segments")

## 8. Load as LlamaIndex `Document`s

In [ ]:
docs = MeetStreamReader(meeting_id=bot_id, client=client).load_data()
for doc in docs[:3]:
    print(doc.metadata["speaker"], "|", doc.text)

## 9. The tools

Called directly, with no LLM. Read-only tools only.

In [ ]:
tools = {t.metadata.name: t for t in get_meetstream_tools(client)}
for name, t in tools.items():
    print(f"{name}: {t.metadata.description.splitlines()[0]}")
print(tools["get_meeting"].call(meeting_id=bot_id).raw_output)
print(len(tools["get_meeting_transcript"].call(meeting_id=bot_id).raw_output["segments"]), "segments via the tool")

## 10. An agent with MeetStream tools

In [ ]:
if RUN_LLM_CELLS:
    from llama_index.core.agent.workflow import FunctionAgent
    from llama_index.llms.openai import OpenAI

    agent = FunctionAgent(tools=list(tools.values()), llm=OpenAI(model="gpt-4o-mini"))
    print(await agent.run(f"Summarize meeting {bot_id} in three bullets."))

## 11. RAG over the transcript

In [ ]:
if RUN_LLM_CELLS and docs:
    from llama_index.core import VectorStoreIndex

    index = VectorStoreIndex.from_documents(docs)
    print(index.as_query_engine().query("What was decided?"))

## 12. Errors

Every failure is a `MeetStreamAPIError` with a stable `code`:

In [ ]:
try:
    client.get_meeting("00000000-0000-0000-0000-000000000000")
except MeetStreamAPIError as exc:
    print(exc.code, exc.status_code, exc)